# ASR 모델 평가 (Evaluation)

## 목적

동일한 데이터셋에서 ASR 모델을 일관된 조건으로 평가하고 결과를 기록하기
위한 노트북입니다.

평가 대상 - Hugging Face 기본 모델 - LoRA Fine-tuned 모델 - Full
Fine-tuned 모델

## 입력

-   Hugging Face Dataset (`train`, `development`, `validation`)
-   원본 ASR TSV (메타데이터 확인용)

## 수행 과정

1.  평가 환경 설정
2.  모델 설정 검증
3.  Dataset 로딩
4.  모델 및 Processor 로딩
5.  음성인식 수행
6.  WER/CER 계산
7.  결과 저장
8.  오류 샘플 확인 및 음성 재생

## 저장 결과

평가를 수행할 때마다 새로운 실행 폴더가 생성되며 기존 결과는 덮어쓰지
않습니다.

생성 파일 - `config.json` : 실험 설정 - `environment.json` : 실행 환경 -
`dataset_metadata.json` : 데이터셋 정보 - `model_metadata.json` : 모델
정보 - `detail.csv` : 파일별 인식 결과 - `summary.csv` : 평가 결과
요약 - `errors.csv` : 오류 정보 - `worst_cases.csv` : CER가 높은 샘플 -
`SUCCESS` / `FAILED` : 실행 상태

## 평가 지표

-   WER
-   CER
-   Macro WER / CER
-   처리 시간
-   Real-Time Factor (RTF)

## 활용

생성된 결과는 이후 모델 비교, 파인튜닝 전후 비교, 추론 옵션 비교 등을
위한 공통 입력으로 사용합니다.


## 1. 패키지 설치

In [ ]:
!pip install -U transformers datasets accelerate peft jiwer pandas tqdm

## 2. 라이브러리 로드

In [ ]:
from pathlib import Path
from datetime import datetime
import gc, hashlib, json, os, platform, random, re, shutil, socket, sys, time, traceback, uuid

import numpy as np
import pandas as pd
import torch
import datasets
import transformers

from datasets import load_from_disk
from jiwer import wer, cer
from tqdm.auto import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

try:
    import peft
    from peft import PeftConfig, PeftModel
except ImportError:
    peft = PeftConfig = PeftModel = None

## 3. 평가 설정 (수정 필요)

In [ ]:
# 평가 정보
EXPERIMENTER, EXPERIMENT_TAG = "yroh", "baseline"
DATASET_NAME, DATASET_SPLIT = "aihub186", "validation"  # train / development / validation

# 모델 설정: base / full_finetuned / peft_lora
MODEL_SOURCE_TYPE = "base"
BASE_MODEL_ID = "openai/whisper-tiny"

FULL_MODEL_DIR = None
LORA_ADAPTER_DIR = None
FINETUNE_RUN_DIR = None
PROCESSOR_SOURCE = None  # 자동: FINETUNE_RUN_DIR/processor → FULL_MODEL_DIR → BASE_MODEL_ID

# 데이터 및 결과 경로
ASR_TSV = Path("/home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv")
HF_DATASET_DIR = Path("/home/data/expr/week2/04-asr_aihub/hf_dataset_whisper-tiny")
RESULTS_ROOT = Path("/home/data/expr/week2/05-asr_evaluation_results")

# 추론 설정
LANGUAGE, TASK = "ko", "transcribe"
BATCH_SIZE, MAX_FILES, SEED = 64, 0, 42  # MAX_FILES: 0/None이면 전체
MAX_NEW_TOKENS, NUM_BEAMS = 128, 1
DO_SAMPLE, USE_FP16, USE_SAFETENSORS = False, True, True

# 오류 처리
RETRY_FAILED_BATCH_INDIVIDUALLY = True
CONTINUE_ON_ERROR = True

# LoRA 평가
#
# LMODEL_SOURCE_TYPE = "peft_lora"
# LFINETUNE_RUN_DIR = Path("/path/to/finetune_run")

# 전체 파인튜닝 모델
#
# MODEL_SOURCE_TYPE = "full_finetuned"
# FINETUNE_RUN_DIR = Path("/path/to/finetune_run")

## 4. 공통 함수

In [ ]:
def normalize_optional_path(value):
    value = "" if value is None else str(value).strip()
    return Path(value).expanduser().resolve() if value else None


def read_json_file(path):
    if path is None:
        return None

    path = Path(path)
    if not path.is_file():
        return None

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# 기존 이름과 호환
read_json_if_exists = read_json_file


def write_json_file(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)


def find_model_weight_files(model_dir):
    model_dir = Path(model_dir)
    names = {
        "model.safetensors",
        "pytorch_model.bin",
        "model.safetensors.index.json",
        "pytorch_model.bin.index.json",
    }
    return [model_dir / name for name in names if (model_dir / name).is_file()]


def extract_base_model_from_metadata(metadata):
    if not isinstance(metadata, dict):
        return None

    keys = ("base_model_id", "base_model_name_or_path", "model_id")
    sources = [metadata.get("model"), metadata]

    for source in sources:
        if not isinstance(source, dict):
            continue

        for key in keys:
            value = source.get(key)
            if value:
                return str(value).strip()

    return None


def safe_slug(value):
    value = re.sub(r"[^0-9A-Za-z가-힣._+-]+", "_", str(value).strip())
    return value.strip("._-") or "unknown"


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    if not path.is_file():
        return None

    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


def safe_text(value):
    return "" if value is None else str(value).strip()


def safe_wer(reference, hypothesis):
    try:
        return float(wer([safe_text(reference)], [safe_text(hypothesis)]))
    except Exception:
        return np.nan


def safe_cer(reference, hypothesis):
    try:
        return float(cer([safe_text(reference)], [safe_text(hypothesis)]))
    except Exception:
        return np.nan


def directory_file_manifest(root_dir, max_hash_size_mb=100):
    if root_dir is None:
        return None

    root_dir = Path(root_dir)
    if not root_dir.is_dir():
        return None

    max_hash_bytes = max_hash_size_mb * 1024 * 1024
    files = []

    for path in sorted(p for p in root_dir.rglob("*") if p.is_file()):
        stat = path.stat()

        files.append({
            "relative_path": str(path.relative_to(root_dir)),
            "size_bytes": stat.st_size,
            "modified_at": datetime.fromtimestamp(stat.st_mtime).astimezone().isoformat(),
            "sha256": file_sha256(path) if stat.st_size <= max_hash_bytes else None,
        })

    return {
        "root": str(root_dir),
        "files": files,
    }
def to_dict_safe(obj):
    try:
        return obj.to_dict()
    except Exception as exc:
        return {"read_error": str(exc)}


## 5. 모델 설정 검증 및 실제 경로 결정

In [ ]:
# ============================================================
# 설정 검증 및 경로 결정
# ============================================================

VALID_MODEL_SOURCE_TYPES = {"base", "full_finetuned", "peft_lora"}

MODEL_SOURCE_TYPE = str(MODEL_SOURCE_TYPE).strip().lower()
BASE_MODEL_ID = "" if BASE_MODEL_ID is None else str(BASE_MODEL_ID).strip()

if MODEL_SOURCE_TYPE not in VALID_MODEL_SOURCE_TYPES:
    raise ValueError(f"MODEL_SOURCE_TYPE 오류: {MODEL_SOURCE_TYPE}")

if not EXPERIMENTER.strip():
    raise ValueError("EXPERIMENTER가 필요합니다.")

if not EXPERIMENT_TAG.strip():
    raise ValueError("EXPERIMENT_TAG가 필요합니다.")

if DATASET_SPLIT not in {"train", "development", "validation"}:
    raise ValueError("DATASET_SPLIT 오류")

FULL_MODEL_DIR = normalize_optional_path(FULL_MODEL_DIR)
LORA_ADAPTER_DIR = normalize_optional_path(LORA_ADAPTER_DIR)
FINETUNE_RUN_DIR = normalize_optional_path(FINETUNE_RUN_DIR)
PROCESSOR_SOURCE = str(PROCESSOR_SOURCE).strip() if PROCESSOR_SOURCE else None

# ------------------------------------------------------------
# 파인튜닝 메타데이터
# ------------------------------------------------------------

finetune_metadata = None
finetune_dataset_manifest = None
finetune_training_args = None
finetune_environment = None
finetune_metrics = None

if FINETUNE_RUN_DIR:
    if not FINETUNE_RUN_DIR.is_dir():
        raise FileNotFoundError(FINETUNE_RUN_DIR)

    finetune_metadata = read_json_file(FINETUNE_RUN_DIR / "metadata.json")
    finetune_dataset_manifest = read_json_file(FINETUNE_RUN_DIR / "dataset_manifest.json")
    finetune_training_args = read_json_file(FINETUNE_RUN_DIR / "training_args.json")
    finetune_environment = read_json_file(FINETUNE_RUN_DIR / "environment.json")
    finetune_metrics = read_json_file(FINETUNE_RUN_DIR / "metrics.json")


# ------------------------------------------------------------
# 모델 경로 자동 결정
# ------------------------------------------------------------

if MODEL_SOURCE_TYPE == "base":

    if any([FULL_MODEL_DIR, LORA_ADAPTER_DIR, FINETUNE_RUN_DIR]):
        raise ValueError("base 모델에서는 파인튜닝 경로를 사용하지 않습니다.")

    resolved_base_model_id = BASE_MODEL_ID
    model_load_source = BASE_MODEL_ID
    model_display_name = BASE_MODEL_ID.replace("/", "_")

elif MODEL_SOURCE_TYPE == "full_finetuned":

    if FULL_MODEL_DIR is None and FINETUNE_RUN_DIR:
        FULL_MODEL_DIR = FINETUNE_RUN_DIR / "full_model"

    if not FULL_MODEL_DIR or not FULL_MODEL_DIR.is_dir():
        raise FileNotFoundError("FULL_MODEL_DIR")

    resolved_base_model_id = extract_base_model_from_metadata(finetune_metadata) or BASE_MODEL_ID
    model_load_source = str(FULL_MODEL_DIR)
    model_display_name = f"full_{safe_slug(FULL_MODEL_DIR.parent.name)}"

elif MODEL_SOURCE_TYPE == "peft_lora":

    if LORA_ADAPTER_DIR is None and FINETUNE_RUN_DIR:
        LORA_ADAPTER_DIR = FINETUNE_RUN_DIR / "lora_adapter"

    if not LORA_ADAPTER_DIR or not LORA_ADAPTER_DIR.is_dir():
        raise FileNotFoundError("LORA_ADAPTER_DIR")

    adapter = read_json_file(LORA_ADAPTER_DIR / "adapter_config.json")

    resolved_base_model_id = (
        adapter.get("base_model_name_or_path")
        or extract_base_model_from_metadata(finetune_metadata)
        or BASE_MODEL_ID
    )

    model_load_source = str(LORA_ADAPTER_DIR)
    model_display_name = f"lora_{safe_slug(LORA_ADAPTER_DIR.parent.name)}"


# ------------------------------------------------------------
# Processor 자동 결정
# ------------------------------------------------------------

if PROCESSOR_SOURCE:
    resolved_processor_source = PROCESSOR_SOURCE

elif FINETUNE_RUN_DIR and (FINETUNE_RUN_DIR / "processor").is_dir():
    resolved_processor_source = str(FINETUNE_RUN_DIR / "processor")

elif MODEL_SOURCE_TYPE == "full_finetuned":
    resolved_processor_source = str(FULL_MODEL_DIR)

else:
    resolved_processor_source = resolved_base_model_id


print("=" * 80)
print(f"Model Type : {MODEL_SOURCE_TYPE}")
print(f"Base Model : {resolved_base_model_id}")
print(f"Model Path : {model_load_source}")
print(f"Processor  : {resolved_processor_source}")
print("=" * 80)

## 6. 고유 평가 실행 폴더 생성

In [ ]:
started_at = datetime.now().astimezone()
RUN_ID = "_".join([
    started_at.strftime("%Y%m%dT%H%M%S%z"),
    safe_slug(EXPERIMENTER),
    safe_slug(EXPERIMENT_TAG),
    uuid.uuid4().hex[:8],
])

RUN_DIR = (
    RESULTS_ROOT
    / safe_slug(DATASET_NAME)
    / safe_slug(model_display_name)
    / safe_slug(DATASET_SPLIT)
    / started_at.strftime("%Y-%m-%d")
    / RUN_ID
)

RUN_DIR.mkdir(parents=True, exist_ok=False)

RUNNING_FILE, SUCCESS_FILE, FAILED_FILE = (
    RUN_DIR / "RUNNING",
    RUN_DIR / "SUCCESS",
    RUN_DIR / "FAILED",
)

write_json_file(RUNNING_FILE, {
    "run_id": RUN_ID,
    "status": "running",
    "started_at": started_at.isoformat(),
})

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)

## 7. 평가 설정 저장

In [ ]:
evaluation_config = {
    "schema_version": "1.0",
    "run": {
        "run_id": RUN_ID,
        "started_at": started_at.isoformat(),
        "experimenter": EXPERIMENTER,
        "experiment_tag": EXPERIMENT_TAG,
    },
    "model": {
        "source_type": MODEL_SOURCE_TYPE,
        "display_name": model_display_name,
        "base_model_id": resolved_base_model_id,
        "model_load_source": str(model_load_source),
        "processor_source": str(resolved_processor_source),
        "full_model_dir": str(FULL_MODEL_DIR) if FULL_MODEL_DIR else None,
        "lora_adapter_dir": str(LORA_ADAPTER_DIR) if LORA_ADAPTER_DIR else None,
        "finetune_run_dir": str(FINETUNE_RUN_DIR) if FINETUNE_RUN_DIR else None,
    },
    "dataset": {
        "dataset_name": DATASET_NAME,
        "hf_dataset_dir": str(HF_DATASET_DIR),
        "split": DATASET_SPLIT,
        "asr_tsv": str(ASR_TSV),
    },
    "generation": {
        "language": LANGUAGE,
        "task": TASK,
        "batch_size": BATCH_SIZE,
        "max_files": MAX_FILES,
        "seed": SEED,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "do_sample": DO_SAMPLE,
    },
    "runtime": {
        "use_fp16": USE_FP16,
        "use_safetensors": USE_SAFETENSORS,
        "retry_failed_batch_individually": RETRY_FAILED_BATCH_INDIVIDUALLY,
        "continue_on_error": CONTINUE_ON_ERROR,
    },
}

write_json_file(RUN_DIR / "config.json", evaluation_config)
print(json.dumps(evaluation_config, ensure_ascii=False, indent=2))

## 8. 실행 환경 저장

In [ ]:
cuda_available = torch.cuda.is_available()

environment = {
    "hostname": socket.gethostname(),
    "platform": platform.platform(),
    "python_version": sys.version,
    "libraries": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "peft": peft.__version__ if peft else None,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "cuda": {
        "available": cuda_available,
        "torch_cuda_version": torch.version.cuda,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES", ""),
        "device_count": torch.cuda.device_count() if cuda_available else 0,
        "device_names": [
            torch.cuda.get_device_name(i)
            for i in range(torch.cuda.device_count())
        ] if cuda_available else [],
    },
}

write_json_file(RUN_DIR / "environment.json", environment)

## 9. 파인튜닝 메타데이터 스냅샷 저장

In [ ]:
finetune_snapshot = {
    "finetune_run_dir": str(FINETUNE_RUN_DIR) if FINETUNE_RUN_DIR else None,
    "metadata": finetune_metadata,
    "dataset_manifest": finetune_dataset_manifest,
    "training_args": finetune_training_args,
    "environment": finetune_environment,
    "metrics": finetune_metrics,
}

write_json_file(RUN_DIR / "finetune_metadata_snapshot.json", finetune_snapshot)

## 10. 모델 파일 메타데이터 저장

In [ ]:
model_artifact_manifest = {
    "source_type": MODEL_SOURCE_TYPE,
    "base_model_id": resolved_base_model_id,
    "full_model": directory_file_manifest(FULL_MODEL_DIR) if FULL_MODEL_DIR else None,
    "lora_adapter": directory_file_manifest(LORA_ADAPTER_DIR) if LORA_ADAPTER_DIR else None,
}

write_json_file(RUN_DIR / "model_artifacts.json", model_artifact_manifest)

## 11. Dataset 읽기

In [ ]:
SPLIT_DIR = HF_DATASET_DIR / DATASET_SPLIT

if not HF_DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset 루트 없음: {HF_DATASET_DIR}")
if not SPLIT_DIR.is_dir():
    raise FileNotFoundError(f"평가 split 없음: {SPLIT_DIR}")

dataset = load_from_disk(str(SPLIT_DIR))

required_columns = {"file_id", "duration_sec", "input_features", "labels"}
missing = required_columns - set(dataset.column_names)

if missing:
    raise ValueError(f"Dataset 열 누락: {sorted(missing)}")
if not len(dataset):
    raise ValueError("평가 Dataset이 비어 있습니다.")

print(dataset)
print("columns:", dataset.column_names)

## 12. Dataset 메타데이터 저장

In [ ]:
dataset_total_seconds = sum(map(float, dataset["duration_sec"]))

dataset_metadata = {
    "dataset_name": DATASET_NAME,
    "dataset_root": str(HF_DATASET_DIR),
    "split": DATASET_SPLIT,
    "split_path": str(SPLIT_DIR),
    "num_rows": len(dataset),
    "columns": dataset.column_names,
    "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
    "total_seconds": dataset_total_seconds,
    "total_hours": dataset_total_seconds / 3600,
    "source_tsv": str(ASR_TSV),
    "source_tsv_exists": ASR_TSV.is_file(),
    "source_tsv_sha256": file_sha256(ASR_TSV) if ASR_TSV.is_file() else None,
}

write_json_file(RUN_DIR / "dataset_metadata.json", dataset_metadata)
display(pd.DataFrame([dataset_metadata]).T)

## 13. 평가 샘플 선택

In [ ]:
total_dataset_count = len(dataset)

if MAX_FILES and MAX_FILES < total_dataset_count:
    selected_indices = sorted(random.Random(SEED).sample(range(total_dataset_count), MAX_FILES))
    eval_dataset = dataset.select(selected_indices)
else:
    selected_indices = list(range(total_dataset_count))
    eval_dataset = dataset

selected_samples_df = pd.DataFrame({
    "dataset_index": selected_indices,
    "file_id": eval_dataset["file_id"],
    "duration_sec": eval_dataset["duration_sec"],
})

selected_samples_df.to_csv(
    RUN_DIR / "selected_samples.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"전체 Dataset: {total_dataset_count:,}")
print(f"평가 대상: {len(eval_dataset):,}")
display(selected_samples_df.head(10))

## 14. 원본 TSV 메타데이터 읽기

In [ ]:
if not ASR_TSV.is_file():
    raise FileNotFoundError(f"ASR TSV 없음: {ASR_TSV}")

source_df = pd.read_csv(
    ASR_TSV,
    sep="\t",
    encoding="utf-8-sig",
    dtype={"file_id": str, "audio_path": str, "transcript": str, "dataset_type": str},
)

source_df.columns = source_df.columns.str.replace("\ufeff", "", regex=False).str.strip()

required_columns = {"file_id", "audio_path", "transcript", "duration_sec", "dataset_type"}
missing = required_columns - set(source_df.columns)

if missing:
    raise ValueError(f"ASR TSV 열 누락: {sorted(missing)}")

text_columns = ["file_id", "audio_path", "transcript", "dataset_type"]

for column in text_columns:
    source_df[column] = source_df[column].fillna("").astype(str).str.strip()

source_df["duration_sec"] = pd.to_numeric(source_df["duration_sec"], errors="coerce")

duplicate_count = source_df.duplicated("file_id", keep=False).sum()
if duplicate_count:
    print(f"주의: 중복 file_id {duplicate_count:,}행")

source_map_df = source_df.drop_duplicates("file_id", keep="first").copy()

print(f"Source metadata rows: {len(source_map_df):,}")

## 15. Processor와 모델 로딩

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.float16 if DEVICE.type == "cuda" and USE_FP16 else torch.float32

print("DEVICE:", DEVICE)
print("MODEL_DTYPE:", MODEL_DTYPE)

processor = AutoProcessor.from_pretrained(
    resolved_processor_source,
    language=LANGUAGE,
    task=TASK,
)

model_source = (
    resolved_base_model_id
    if MODEL_SOURCE_TYPE == "base"
    else str(FULL_MODEL_DIR)
    if MODEL_SOURCE_TYPE == "full_finetuned"
    else resolved_base_model_id
)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_source,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=USE_SAFETENSORS,
)

if MODEL_SOURCE_TYPE == "peft_lora":
    if PeftModel is None:
        raise ImportError("peft가 설치되어 있지 않습니다.")
    model = PeftModel.from_pretrained(
        model,
        str(LORA_ADAPTER_DIR),
        is_trainable=False,
    )

model = model.to(DEVICE).eval()

if hasattr(model, "generation_config"):
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.do_sample = DO_SAMPLE
    model.generation_config.num_beams = NUM_BEAMS

print("Model loaded:", type(model).__name__)
print("Processor:", type(processor).__name__)

## 16. 로드된 모델 메타데이터 저장

In [ ]:
config_object = model.get_base_model().config if MODEL_SOURCE_TYPE == "peft_lora" else model.config
model_config = to_dict_safe(config_object)
generation_config = to_dict_safe(model.generation_config)

peft_config = None
if MODEL_SOURCE_TYPE == "peft_lora":
    try:
        peft_config = {name: config.to_dict() for name, config in model.peft_config.items()}
    except Exception as exc:
        peft_config = {"read_error": str(exc)}

loaded_model_metadata = {
    "source_type": MODEL_SOURCE_TYPE,
    "model_class": type(model).__name__,
    "base_model_id": resolved_base_model_id,
    "model_load_source": str(model_load_source),
    "processor_source": str(resolved_processor_source),
    "dtype": str(MODEL_DTYPE),
    "device": str(DEVICE),
    "model_config": model_config,
    "generation_config": generation_config,
    "peft_config": peft_config,
}

write_json_file(RUN_DIR / "model_metadata.json", loaded_model_metadata)

## 17. 추론 함수

In [ ]:
def decode_reference(label_ids):
    label_ids = np.asarray(label_ids, dtype=np.int64).copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    return safe_text(processor.tokenizer.decode(label_ids, skip_special_tokens=True))


@torch.inference_mode()
def generate_batch(examples):
    input_features = torch.stack([
        torch.as_tensor(ex["input_features"], dtype=MODEL_DTYPE)
        for ex in examples
    ]).to(DEVICE)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    started = time.perf_counter()

    generated_ids = model.generate(
        input_features=input_features,
        language=LANGUAGE,
        task=TASK,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=DO_SAMPLE,
    )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    hypotheses = processor.tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return hypotheses, time.perf_counter() - started

## 18. 평가 실행

In [ ]:
detail_rows, error_rows = [], []
all_references, all_hypotheses = [], []
total_audio_seconds = 0.0

evaluation_started_at = datetime.now().astimezone()
evaluation_start_perf = time.perf_counter()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def append_success_result(example, hypothesis, batch_index, elapsed, mode):
    reference = decode_reference(example["labels"])
    hypothesis = safe_text(hypothesis)

    detail_rows.append({
        "run_id": RUN_ID,
        "experimenter": EXPERIMENTER,
        "experiment_tag": EXPERIMENT_TAG,
        "dataset_name": DATASET_NAME,
        "dataset_split": DATASET_SPLIT,
        "model_source_type": MODEL_SOURCE_TYPE,
        "model_display_name": model_display_name,
        "base_model_id": resolved_base_model_id,
        "finetune_run_dir": str(FINETUNE_RUN_DIR) if FINETUNE_RUN_DIR else None,
        "file_id": example["file_id"],
        "duration_sec": float(example["duration_sec"]),
        "reference": reference,
        "hypothesis": hypothesis,
        "wer": safe_wer(reference, hypothesis),
        "cer": safe_cer(reference, hypothesis),
        "batch_index": batch_index,
        "inference_mode": mode,
        "inference_elapsed_sec": elapsed,
    })

    all_references.append(reference)
    all_hypotheses.append(hypothesis)


def append_error_result(example, batch_index, exception, mode):
    error_rows.append({
        "run_id": RUN_ID,
        "model_source_type": MODEL_SOURCE_TYPE,
        "base_model_id": resolved_base_model_id,
        "file_id": example.get("file_id", ""),
        "batch_index": batch_index,
        "inference_mode": mode,
        "error_type": type(exception).__name__,
        "error_message": str(exception),
        "traceback": traceback.format_exc(),
    })


batch_ranges = range(0, len(eval_dataset), BATCH_SIZE)
total_batches = (len(eval_dataset) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_index, batch_start in enumerate(
    tqdm(batch_ranges, total=total_batches, desc="ASR evaluation")
):
    batch_end = min(batch_start + BATCH_SIZE, len(eval_dataset))
    examples = [eval_dataset[i] for i in range(batch_start, batch_end)]
    total_audio_seconds += sum(float(ex["duration_sec"]) for ex in examples)

    try:
        hypotheses, batch_elapsed = generate_batch(examples)
        per_file_elapsed = batch_elapsed / len(examples)

        for example, hypothesis in zip(examples, hypotheses):
            append_success_result(
                example,
                hypothesis,
                batch_index,
                per_file_elapsed,
                "batch",
            )

    except Exception as batch_exception:
        print(
            f"\n[배치 오류] {batch_start}~{batch_end - 1}: "
            f"{type(batch_exception).__name__}: {batch_exception}"
        )

        if RETRY_FAILED_BATCH_INDIVIDUALLY:
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

            for example in examples:
                try:
                    hypotheses, elapsed = generate_batch([example])
                    append_success_result(
                        example,
                        hypotheses[0],
                        batch_index,
                        elapsed,
                        "individual_retry",
                    )
                except Exception as file_exception:
                    append_error_result(
                        example,
                        batch_index,
                        file_exception,
                        "individual_retry",
                    )
                    if not CONTINUE_ON_ERROR:
                        raise
        else:
            for example in examples:
                append_error_result(
                    example,
                    batch_index,
                    batch_exception,
                    "batch",
                )

            if not CONTINUE_ON_ERROR:
                raise


evaluation_elapsed_sec = time.perf_counter() - evaluation_start_perf
evaluation_finished_at = datetime.now().astimezone()

print(f"정상 평가: {len(detail_rows):,}")
print(f"평가 오류: {len(error_rows):,}")
print(f"평가 시간: {evaluation_elapsed_sec:,.2f}초")

## 19. 상세 결과와 원본 메타데이터 결합

In [ ]:
detail_df, errors_df = pd.DataFrame(detail_rows), pd.DataFrame(error_rows)

if detail_df.empty:
    raise RuntimeError("정상 평가 결과가 없습니다. 오류 내용을 확인하세요.")

detail_df = (
    detail_df
    .merge(
        source_map_df[["file_id", "audio_path", "transcript", "dataset_type"]],
        on="file_id",
        how="left",
    )
    .rename(columns={
        "transcript": "source_transcript",
        "dataset_type": "source_dataset_type",
    })
)

detail_columns = [
    "run_id", "experimenter", "experiment_tag",
    "dataset_name", "dataset_split",
    "model_source_type", "model_display_name", "base_model_id", "finetune_run_dir",
    "file_id", "audio_path", "duration_sec", "source_dataset_type",
    "source_transcript", "reference", "hypothesis",
    "wer", "cer", "batch_index", "inference_mode", "inference_elapsed_sec",
]

detail_df = detail_df[[c for c in detail_columns if c in detail_df.columns]].copy()

display(detail_df.head(10))

## 20. 전체 평가 지표 계산

In [ ]:
overall_wer = float(wer(all_references, all_hypotheses))
overall_cer = float(cer(all_references, all_hypotheses))
macro_wer = float(detail_df["wer"].mean())
macro_cer = float(detail_df["cer"].mean())

n_success, n_error = len(detail_df), len(errors_df)
sec_per_file = evaluation_elapsed_sec / max(n_success, 1)
files_per_sec = n_success / max(evaluation_elapsed_sec, 1e-9)
real_time_factor = evaluation_elapsed_sec / max(total_audio_seconds, 1e-9)
peak_gpu_memory_gb = (
    float(torch.cuda.max_memory_allocated() / 1024**3)
    if DEVICE.type == "cuda"
    else None
)

finetune_run_id = (
    finetune_metadata.get("run_id")
    if isinstance(finetune_metadata, dict)
    else None
)

summary_row = {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "started_at": evaluation_started_at.isoformat(),
    "finished_at": evaluation_finished_at.isoformat(),
    "experimenter": EXPERIMENTER,
    "experiment_tag": EXPERIMENT_TAG,

    "dataset_name": DATASET_NAME,
    "dataset_split": DATASET_SPLIT,
    "dataset_dir": str(HF_DATASET_DIR),
    "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
    "source_tsv_sha256": file_sha256(ASR_TSV),

    "model_source_type": MODEL_SOURCE_TYPE,
    "model_display_name": model_display_name,
    "base_model_id": resolved_base_model_id,
    "model_load_source": str(model_load_source),
    "processor_source": str(resolved_processor_source),
    "finetune_run_id": finetune_run_id,
    "finetune_run_dir": str(FINETUNE_RUN_DIR) if FINETUNE_RUN_DIR else None,
    "lora_adapter_dir": str(LORA_ADAPTER_DIR) if LORA_ADAPTER_DIR else None,
    "full_model_dir": str(FULL_MODEL_DIR) if FULL_MODEL_DIR else None,

    "n_dataset_total": total_dataset_count,
    "n_selected": len(eval_dataset),
    "n_success": n_success,
    "n_error": n_error,

    "total_audio_seconds": total_audio_seconds,
    "total_audio_hours": total_audio_seconds / 3600,
    "elapsed_sec": evaluation_elapsed_sec,
    "sec_per_file": sec_per_file,
    "files_per_sec": files_per_sec,
    "real_time_factor": real_time_factor,

    "wer": overall_wer,
    "cer": overall_cer,
    "macro_wer": macro_wer,
    "macro_cer": macro_cer,

    "batch_size": BATCH_SIZE,
    "max_files": MAX_FILES,
    "seed": SEED,
    "language": LANGUAGE,
    "task": TASK,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": NUM_BEAMS,
    "do_sample": DO_SAMPLE,

    "device": str(DEVICE),
    "dtype": str(MODEL_DTYPE),
    "peak_gpu_memory_gb": peak_gpu_memory_gb,

    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "peft_version": peft.__version__ if peft else None,
    "torch_version": torch.__version__,
}

summary_df = pd.DataFrame([summary_row])
display(summary_df.T)

## 21. 결과 저장

In [ ]:
detail_file = RUN_DIR / "detail.csv"
summary_file = RUN_DIR / "summary.csv"
errors_file = RUN_DIR / "errors.csv"
worst_cases_file = RUN_DIR / "worst_cases.csv"

detail_df.to_csv(detail_file, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
errors_df.to_csv(errors_file, index=False, encoding="utf-8-sig")

worst_cases_df = detail_df.sort_values(
    ["cer", "wer"],
    ascending=False,
).reset_index(drop=True)

worst_cases_df.to_csv(
    worst_cases_file,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", summary_file)
print("Saved:", detail_file)
print("Saved:", errors_file)
print("Saved:", worst_cases_file)

## 22. 실행 완료 표시

In [ ]:
completion_data = {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "status": "success",
    "started_at": evaluation_started_at.isoformat(),
    "finished_at": evaluation_finished_at.isoformat(),
    "experimenter": EXPERIMENTER,
    "experiment_tag": EXPERIMENT_TAG,
    "model_source_type": MODEL_SOURCE_TYPE,
    "model_display_name": model_display_name,
    "base_model_id": resolved_base_model_id,
    "finetune_run_id": finetune_run_id,
    "dataset_name": DATASET_NAME,
    "dataset_split": DATASET_SPLIT,
    "n_success": n_success,
    "n_error": n_error,
    "wer": overall_wer,
    "cer": overall_cer,
    "summary_file": str(summary_file),
    "detail_file": str(detail_file),
}

write_json_file(SUCCESS_FILE, completion_data)

if RUNNING_FILE.exists():
    RUNNING_FILE.unlink()

EXPERIMENTS_MD = RESULTS_ROOT / "EXPERIMENTS.md"

if not EXPERIMENTS_MD.exists():
    EXPERIMENTS_MD.write_text(
        "# ASR Evaluation Experiments\n\n"
        "| Finished | Run ID | Experimenter | Dataset | Split | Model | Type | Files | WER | CER | RTF | Batch | Result |\n"
        "|---|---|---|---|---|---|---|---:|---:|---:|---:|---:|---|\n",
        encoding="utf-8",
    )

result_link = summary_file.relative_to(RESULTS_ROOT).as_posix()

with EXPERIMENTS_MD.open("a", encoding="utf-8") as f:
    f.write(
        f"| {evaluation_finished_at.strftime('%Y-%m-%d %H:%M:%S')} "
        f"| {RUN_ID} "
        f"| {EXPERIMENTER} "
        f"| {DATASET_NAME} "
        f"| {DATASET_SPLIT} "
        f"| {model_display_name} "
        f"| {MODEL_SOURCE_TYPE} "
        f"| {n_success:,} "
        f"| {overall_wer:.6f} "
        f"| {overall_cer:.6f} "
        f"| {real_time_factor:.6f} "
        f"| {BATCH_SIZE} "
        f"| [summary]({result_link}) |\n"
    )

print("=" * 80)
print(f"평가 완료 | RUN_ID: {RUN_ID}")
print(f"RUN_DIR: {RUN_DIR}")
print(f"MODEL: {model_display_name}")
print(f"WER: {overall_wer:.6f} | CER: {overall_cer:.6f} | RTF: {real_time_factor:.6f}")
print(f"INDEX: {EXPERIMENTS_MD}")
print("=" * 80)

## 23. 오류가 큰 샘플 재생

In [ ]:
from IPython.display import Audio, display

REVIEW_COUNT = 10
review_df = detail_df.sort_values(["cer", "wer"], ascending=False).head(REVIEW_COUNT)

for i, row in enumerate(review_df.itertuples(index=False), 1):
    audio_path = Path(str(row.audio_path))

    print("=" * 100)
    print(f"[{i}/{len(review_df)}] {row.file_id}")
    print(f"MODEL: {row.model_display_name} ({row.model_source_type})")
    print(f"DURATION: {row.duration_sec} sec | CER: {row.cer:.6f} | WER: {row.wer:.6f}")
    print("REF:", row.reference)
    print("HYP:", row.hypothesis)
    print("PATH:", audio_path)

    if audio_path.is_file():
        display(Audio(filename=str(audio_path)))
    else:
        print("[ERROR] 음성 파일을 찾을 수 없습니다.")